# Мордовкин Максим Михайлович, Тарасова Е.Ю., N4170, Задача №2 "Вектор Шепли"

## Вариант №15
## Условие задачи

Дана кооперативная игра трёх игроков $N = \{1, 2, 3\}$ с характеристической функцией:

$$v(\varnothing) = 0,\quad v(\{1\}) = 4,\quad v(\{2\}) = 1,\quad v(\{3\}) = 2$$

$$v(\{1,2\}) = 8,\quad v(\{1,3\}) = 9,\quad v(\{2,3\}) = 6,\quad v(\{1,2,3\}) = 19$$

**Требуется:**

1. Найти вектор Шепли **двумя способами**.
2. Проверить принадлежность вектора Шепли **C-ядру**.

## Вектор Шепли двумя способами

In [ ]:
import numpy as np
import math
from itertools import permutations, combinations

# N = {1, 2, 3}
# v({1})=4, v({2})=1, v({3})=2
# v({1,2})=8, v({1,3})=9, v({2,3})=6
# v({1,2,3})=19

players = [1, 2, 3]
n = len(players)

# Характеристическая функция: ключ — отсортированный кортеж игроков,
# Значение — выигрыш этой коалиции.
v = {
    (): 0,
    (1,): 4,
    (2,): 1,
    (3,): 2,
    (1, 2): 8,
    (1, 3): 9,
    (2, 3): 6,
    (1, 2, 3): 19,
}

def val(S):
    """Значение характеристической функции для коалиции S."""
    return v[tuple(sorted(S))]


# Способ 1 — прямая формула Шепли
# φ_i = Σ_{S ⊆ N\{i}}  |S|! (n-|S|-1)! / n!  *  [v(S∪{i}) - v(S)]
# Идея: перебираем все подкоалиции S, не содержащие игрока i,
# для каждой считаем маргинальный вклад игрока i:
#   v(S ∪ {i}) - v(S),
# и домножаем на весовой коэффициент, зависящий от размера S.

def shapley_formula(v, players):
    n = len(players)
    phi = {i: 0 for i in players}          # накопитель вкладов по игрокам

    for i in players:                       # для каждого игрока i
        others = [j for j in players if j != i]   # все остальные игроки

        # Перебираем все подкоалиции S из "others" — от пустой до полной
        for k in range(len(others) + 1):
            for S in combinations(others, k):
                S = list(S)

                # Вес по формуле Шепли: |S|! (n-|S|-1)! / n!
                weight = (math.factorial(len(S)) *
                          math.factorial(n - len(S) - 1) /
                          math.factorial(n))

                # Маргинальный вклад: насколько вырастёт выигрыш,
                # если к S добавить игрока i.
                marginal = val(S + [i]) - val(S)

                # Добавляем взвешенный вклад
                phi[i] += weight * marginal
    return phi


# Способ 2 — через перестановки (среднее маргинальных вкладов)
# Для каждой из n! перестановок игроки «входят» в общую коалицию
# по очереди. Вклад игрока = прирост значения коалиции при его входе.
# φ_i = (сумма вкладов по всем перестановкам) / n!

def shapley_permutations(v, players):
    phi = {i: 0 for i in players}
    perms = list(permutations(players))     # все n! перестановок

    for perm in perms:
        coalition = []                      # текущая коалиция (растёт по шагам)

        for i in perm:                      # игроки «входят» в порядке perm
            before = val(coalition)         # выигрыш до входа i
            coalition.append(i)             # добавляем игрока i
            after = val(coalition)          # выигрыш после входа i

            # Вклад игрока i в этой перестановке
            phi[i] += after - before

    # Усредняем по всем перестановкам
    for i in players:
        phi[i] /= len(perms)
    return phi

# Считаем вектор Шепли двумя способами
phi1 = shapley_formula(v, players)
phi2 = shapley_permutations(v, players)

print("Способ 1 (прямая формула Шепли):")
for i in players:
    print(f"  φ_{i} = {phi1[i]:.4f}")

print("\nСпособ 2 (перебор перестановок):")
for i in players:
    print(f"  φ_{i} = {phi2[i]:.4f}")

phi = np.array([phi1[i] for i in players])
print("\nВектор Шепли:", phi)
print("Сумма:", phi.sum(), " (должна равняться v(N) =", val(players), ")")

Способ 1 (прямая формула Шепли):
  φ_1 = 8.0000
  φ_2 = 5.0000
  φ_3 = 6.0000

Способ 2 (перебор перестановок):
  φ_1 = 8.0000
  φ_2 = 5.0000
  φ_3 = 6.0000

Вектор Шепли: [8. 5. 6.]
Сумма: 19.0  (должна равняться v(N) = 19 )


## Проверка принадлежности C-ядру

In [ ]:
# Проверка принадлежности вектора Шепли C-ядру
# C-ядро — множество дележей x, для которых:
#   1) Σ x_i = v(N)                     (эффективность)
#   2) Σ_{i∈S} x_i ≥ v(S) для всех S   (устойчивость коалиций)

def check_core(phi, v, players):
    print("Проверка принадлежности C-ядру")
    print("-" * 50)

    # --- Условие эффективности ---
    # Сумма всех компонент дележа должна равняться выигрышу
    # большой коалиции v(N).
    total = sum(phi[i] for i in players)
    vN = val(players)
    ok_eff = np.isclose(total, vN)
    print(f"Эффективность: Σφ = {total} == v(N) = {vN}  ->",
          "OK" if ok_eff else "НАРУШЕНО")

    # --- Условия устойчивости для всех подкоалиций ---
    # Для каждой непустой собственной подкоалиции S проверяем:
    # сумма долей её участников ≥ v(S).
    all_ok = ok_eff
    for r in range(1, len(players)):                # размер коалиции: 1..n-1
        for S in combinations(players, r):          # все подкоалиции размера r
            coalition_sum = sum(phi[i] for i in S)  # Σφ по участникам S
            vS = val(S)                             # выигрыш коалиции S
            ok = coalition_sum >= vS - 1e-9         # допуск на погрешность float
            all_ok &= ok
            print(f"Коалиция {S}: Σφ = {coalition_sum} >= v(S) = {vS}  ->",
                  "OK" if ok else "НАРУШЕНО")

    print("\nВектор Шепли принадлежит C-ядру:", all_ok)
    return all_ok


check_core(phi1, v, players)

Проверка принадлежности C-ядру
--------------------------------------------------
Эффективность: Σφ = 19.0 == v(N) = 19  -> OK
Коалиция (1,): Σφ = 8.0 >= v(S) = 4  -> OK
Коалиция (2,): Σφ = 5.0 >= v(S) = 1  -> OK
Коалиция (3,): Σφ = 6.0 >= v(S) = 2  -> OK
Коалиция (1, 2): Σφ = 13.0 >= v(S) = 8  -> OK
Коалиция (1, 3): Σφ = 14.0 >= v(S) = 9  -> OK
Коалиция (2, 3): Σφ = 11.0 >= v(S) = 6  -> OK

Вектор Шепли принадлежит C-ядру: True


np.True_

## Пояснение

### Способ 1 — прямая формула Шепли

Вектор Шепли вычисляется по формуле:

$$\varphi_i = \sum_{S \subseteq N \setminus \{i\}} \frac{|S|!\,(n-|S|-1)!}{n!}\,\bigl[v(S \cup \{i\}) - v(S)\bigr]$$

Для $n = 3$ веса принимают значения:

$$|S| = 0 \Rightarrow \frac{0!\,2!}{3!} = \frac{1}{3},\qquad
|S| = 1 \Rightarrow \frac{1!\,1!}{3!} = \frac{1}{6},\qquad
|S| = 2 \Rightarrow \frac{2!\,0!}{3!} = \frac{1}{3}$$

**Для игрока 1:**

$$\varphi_1 = \tfrac{1}{3}(4-0) + \tfrac{1}{6}(8-1) + \tfrac{1}{6}(9-2) + \tfrac{1}{3}(19-6) = 8$$

**Для игрока 2:**

$$\varphi_2 = \tfrac{1}{3}(1-0) + \tfrac{1}{6}(8-4) + \tfrac{1}{6}(6-2) + \tfrac{1}{3}(19-9) = 5$$

**Для игрока 3:**

$$\varphi_3 = \tfrac{1}{3}(2-0) + \tfrac{1}{6}(9-4) + \tfrac{1}{6}(6-1) + \tfrac{1}{3}(19-8) = 6$$

### Способ 2 — перебор перестановок

Для каждой из $3! = 6$ перестановок игроки «входят» в коалицию по очереди; вклад игрока равен приросту значения коалиции при его входе. Затем вклады усредняются. Результат совпадает:

$$\varphi = (8,\ 5,\ 6)$$

### Проверка принадлежности C-ядру

Дележ $x$ принадлежит C-ядру, если:

$$\sum_{i \in N} x_i = v(N),\qquad \sum_{i \in S} x_i \ge v(S)\ \ \forall S \subseteq N$$

Проверка для $\varphi = (8,\ 5,\ 6)$:

| Условие | Проверка | Результат |
|---|---|---|
| Эффективность | $8+5+6 = 19 = v(N)$ | ДА |
| Коалиция $\{1\}$ | $8 \ge 4$ | ДА |
| Коалиция $\{2\}$ | $5 \ge 1$ | ДА |
| Коалиция $\{3\}$ | $6 \ge 2$ | ДА |
| Коалиция $\{1,2\}$ | $13 \ge 8$ | ДА |
| Коалиция $\{1,3\}$ | $14 \ge 9$ | ДА |
| Коалиция $\{2,3\}$ | $11 \ge 6$ | ДА |

Все условия выполняются, следовательно, **вектор Шепли принадлежит C-ядру**.

## Ответ

**Вектор Шепли:**

$$\varphi = (\varphi_1,\ \varphi_2,\ \varphi_3) = (8,\ 5,\ 6)$$

**Проверка эффективности:**

$$\varphi_1 + \varphi_2 + \varphi_3 = 8 + 5 + 6 = 19 = v(\{1,2,3\})$$

**Принадлежность C-ядру:** вектор Шепли $\varphi = (8,\ 5,\ 6)$ удовлетворяет условиям эффективности и устойчивости всех коалиций, следовательно, **принадлежит C-ядру**.